In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torchbnn as bnn

In [10]:
# Generate toy 2D data
X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)
scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test  = torch.tensor(y_test, dtype=torch.long)

In [11]:
class SimpleBNN(nn.Module):
    def __init__(self, in_dim=2, hidden_dim=16, out_dim=2):
        super().__init__()
        self.fc1 = bnn.BayesLinear(prior_mu=0, prior_sigma=0.1,
                                   in_features=in_dim, out_features=hidden_dim)
        self.fc2 = bnn.BayesLinear(prior_mu=0, prior_sigma=0.1,
                                   in_features=hidden_dim, out_features=out_dim)
        self.act = nn.ReLU()
        
    def forward(self, x):
        x = self.act(self.fc1(x))
        x = self.fc2(x)
        return x

In [12]:
model = SimpleBNN(in_dim=2, hidden_dim=16, out_dim=2)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# KL regularizer provided by torchbnn
kl_loss = bnn.BKLLoss(reduction='mean', last_layer_only=False)
kl_weight = 0.01

for epoch in range(1000):
    model.train()
    optimizer.zero_grad()

    logits = model(X_train)
    ce = criterion(logits, y_train)

    # KL divergence over Bayesian layers
    kl = kl_loss(model)

    loss = ce + kl_weight * kl
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 50 == 0:
        model.eval()
        with torch.no_grad():
            test_logits = model(X_test)
            preds = test_logits.argmax(dim=1)
            acc = (preds == y_test).float().mean().item()
        print(f"Epoch {epoch+1:3d} | loss={loss.item():.4f} | test_acc={acc:.3f}")

Epoch  50 | loss=0.7429 | test_acc=0.570
Epoch 100 | loss=0.6602 | test_acc=0.530
Epoch 150 | loss=0.5497 | test_acc=0.883
Epoch 200 | loss=0.5097 | test_acc=0.863
Epoch 250 | loss=0.4143 | test_acc=0.883
Epoch 300 | loss=0.4052 | test_acc=0.870
Epoch 350 | loss=0.3780 | test_acc=0.883
Epoch 400 | loss=0.3704 | test_acc=0.877
Epoch 450 | loss=0.3579 | test_acc=0.863
Epoch 500 | loss=0.3555 | test_acc=0.890
Epoch 550 | loss=0.3521 | test_acc=0.897
Epoch 600 | loss=0.3514 | test_acc=0.880
Epoch 650 | loss=0.3474 | test_acc=0.883
Epoch 700 | loss=0.3592 | test_acc=0.877
Epoch 750 | loss=0.3572 | test_acc=0.893
Epoch 800 | loss=0.3445 | test_acc=0.893
Epoch 850 | loss=0.3450 | test_acc=0.890
Epoch 900 | loss=0.3387 | test_acc=0.887
Epoch 950 | loss=0.3412 | test_acc=0.887
Epoch 1000 | loss=0.3452 | test_acc=0.893


In [13]:
torch.manual_seed(0)

N = 1000
# Inputs: x1 in [0,1], x2 in [0,1], x3 in [0,1]
X = torch.rand(N, 3)

x1 = X[:, 0]
x2 = X[:, 1]
x3 = X[:, 2]

eps = 0.05 * torch.randn(N, 2)

y1 = 0.5 + 0.3*x1 - 0.4*x2 + 0.2*x3 + 0.1*x1*x2
y2 = 0.4 + 0.1*x1 + 0.5*x2 - 0.3*x3 + 0.2*(x2**2)

Y = torch.stack([y1, y2], dim=1) + eps  # shape [N, 2]


In [14]:
class BNNMultiRegressor(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, out_dim=3):
        super().__init__()
        self.fc1 = bnn.BayesLinear(prior_mu=0, prior_sigma=0.1,
                                   in_features=in_dim, out_features=hidden_dim)
        self.fc2 = bnn.BayesLinear(prior_mu=0, prior_sigma=0.1,
                                   in_features=hidden_dim, out_features=hidden_dim)
        self.out = bnn.BayesLinear(prior_mu=0, prior_sigma=0.1,
                                   in_features=hidden_dim, out_features=out_dim)
        self.act = nn.ReLU()

    def forward(self, x):
        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))
        y_hat = self.out(x)          # shape [B, out_dim]
        return y_hat    

In [15]:
def predict_mc(model, X, n_samples=50):
    model.eval()
    with torch.no_grad():
        samples = []
        for _ in range(n_samples):
            y_pred = model(X)            # [N, K]
            samples.append(y_pred)
        Y = torch.stack(samples, dim=0)  # [S, N, K]
        mean = Y.mean(0)                 # [N, K]
        std = Y.std(0)                   # [N, K]
    return mean, std

In [16]:
def make_data(N=1000, seed=0):
    torch.manual_seed(seed)
    # Inputs: x1, x2, x3 in [0, 1]
    X = torch.rand(N, 3)
    x1 = X[:, 0]
    x2 = X[:, 1]
    x3 = X[:, 2]

    # Two continuous targets with some nonlinearity + noise
    eps = 0.05 * torch.randn(N, 2)

    y1 = 0.5 + 0.3*x1 - 0.4*x2 + 0.2*x3 + 0.1*x1*x2
    y2 = 0.4 + 0.1*x1 + 0.5*x2 - 0.3*x3 + 0.2*(x2**2)

    Y = torch.stack([y1, y2], dim=1) + eps  # [N, 2]
    return X, Y

def main():
    # 4.1 Data
    X, Y = make_data(N=1000, seed=42)

    X_train, X_test, Y_train, Y_test = train_test_split(
        X.numpy(), Y.numpy(), test_size=0.3, random_state=42
    )

    X_train = torch.tensor(X_train, dtype=torch.float32)
    X_test  = torch.tensor(X_test, dtype=torch.float32)
    Y_train = torch.tensor(Y_train, dtype=torch.float32)
    Y_test  = torch.tensor(Y_test, dtype=torch.float32)

    # 4.2 Model, loss, optimizer
    in_dim = X_train.shape[1]
    out_dim = Y_train.shape[1]
    model = BNNMultiRegressor(in_dim=in_dim, hidden_dim=64, out_dim=out_dim)

    mse = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    # KL regularizer from torchbnn
    kl_loss = bnn.BKLLoss(reduction='mean', last_layer_only=False)
    kl_weight = 0.01

    # 4.3 Training loop
    n_epochs = 300
    for epoch in range(n_epochs):
        model.train()
        optimizer.zero_grad()

        Y_pred = model(X_train)         # [N, 2]
        data_loss = mse(Y_pred, Y_train)
        kl = kl_loss(model)

        loss = data_loss + kl_weight * kl
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 50 == 0:
            model.eval()
            with torch.no_grad():
                Y_test_pred = model(X_test)
                test_loss = mse(Y_test_pred, Y_test).item()
            print(f"Epoch {epoch+1:3d} | train_loss={loss.item():.4f} | test_mse={test_loss:.4f}")

    # 4.4 Evaluation with MC uncertainty
    mean_pred, std_pred = predict_mc(model, X_test, n_samples=100)
    final_mse = mse(mean_pred, Y_test).item()
    print(f"\nFinal MC MSE on test: {final_mse:.4f}")
    print("Example predictions (first 5):")
    for i in range(5):
        print(f"True: {Y_test[i].numpy()}, Pred mean: {mean_pred[i].numpy()}, Pred std: {std_pred[i].numpy()}")
    return model

In [17]:
model = main()

Epoch  50 | train_loss=0.2002 | test_mse=0.2015
Epoch 100 | train_loss=0.1686 | test_mse=0.0746
Epoch 150 | train_loss=0.2440 | test_mse=0.0717
Epoch 200 | train_loss=0.1672 | test_mse=0.1039
Epoch 250 | train_loss=0.1348 | test_mse=0.1782
Epoch 300 | train_loss=0.0844 | test_mse=0.0158

Final MC MSE on test: 0.0114
Example predictions (first 5):
True: [0.46336153 0.657835  ], Pred mean: [0.5028142 0.5758331], Pred std: [0.2309046  0.23839879]
True: [0.75705594 0.47738945], Pred mean: [0.67833483 0.5576602 ], Pred std: [0.259492   0.26296884]
True: [0.67832965 0.7462434 ], Pred mean: [0.6408204  0.72331697], Pred std: [0.28002098 0.2827953 ]
True: [0.61985797 0.58925676], Pred mean: [0.6571808 0.6254717], Pred std: [0.26600122 0.27207372]
True: [0.4933933  0.98867553], Pred mean: [0.58405197 0.8797096 ], Pred std: [0.28875232 0.30288696]


## Sequential Bayesian Neural Network for Dynamic MCDM

In [18]:
class Stage1BNN(nn.Module):    ### R_t(a)
    def __init__(self, in_dim, out_dim, hidden_dim=64):
        super().__init__()
        self.fc1 = bnn.BayesLinear(prior_mu=0, prior_sigma=0.1,
                                   in_features=in_dim, out_features=hidden_dim)
        self.fc2 = bnn.BayesLinear(prior_mu=0, prior_sigma=0.1,
                                   in_features=hidden_dim, out_features=hidden_dim)
        self.out = bnn.BayesLinear(prior_mu=0, prior_sigma=0.1,
                                   in_features=hidden_dim, out_features=out_dim)
        self.act = nn.ReLU()
    
    def forward(self, x):
        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))
        y_hat = self.out(x)          # shape [B, out_dim]

class Stage2BNN(nn.Module):    ### D_E(E_t-1(a), R_t(a))
    def __init__(self, in_dim, out_dim, hidden_dim=64):
        self.fc1 = bnn.BayesLinear(prior_mu=0, prior_sigma=0.1,
                                   in_features=in_dim, out_features=hidden_dim)
        self.fc2 = bnn.BayesLinear(prior_mu=0, prior_sigma=0.1,
                                   in_features=hidden_dim, out_features=hidden_dim)
        self.out = bnn.BayesLinear(prior_mu=0, prior_sigma=0.1,
                                   in_features=hidden_dim, out_features=out_dim)
        self.act = nn.ReLU()
    
    def forward(self, x):
        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))
        y_hat = self.out(x)          # shape [B, out_dim]

class JointBNN(nn.Module):
    def __init__(self, num_criteria, num_alternatives, hidden_dim=64):
        self.bnn1 = Stage1BNN(in_dim=num_criteria, out_dim=num_alternatives, hidden_dim=hidden_dim)
        self.bnn2 = Stage2BNN(in_dim=2*num_criteria, out_dim=num_alternatives, hidden_dim=hidden_dim)
        self.transformer = None ### Insert a pretrained transformer like CHRONOS here (input_dim = num_alternatives)
    
    def feature_select(self):
        ### Implement the function where the features selected will be the only ones fed to the transformer, other features will just be assigned a value of zero 
        pass
    
    def forward(self, x, y_prev):
        x = self.bnn1(x)
        z = torch.cat([x_t, y_prev], dim=-1)
        z = self.bnn2(z)
        ### Insert the transformer here
